In [ ]:

# =====================================================================
# DECODELABS | DATA ANALYTICS PROJECT 1
# DATA CLEANING & PREPARATION
# Google Colab | Complete Single-Cell Project
# =====================================================================

# 1. INSTALLATION
import sys
import subprocess

print("Installing required libraries...")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "openpyxl", "reportlab"
])

# 2. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import os
import re
from datetime import datetime
from google.colab import files

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle
)
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.enums import TA_CENTER

print("All libraries installed and imported successfully!")

# =====================================================================
# 3. PROJECT INTRODUCTION
# =====================================================================

print("\n" + "=" * 70)
print("DECODELABS - DATA ANALYTICS PROJECT 1")
print("DATA CLEANING & PREPARATION")
print("=" * 70)

# =====================================================================
# 4. UPLOAD RAW DATASET
# =====================================================================

print("\nUpload your raw CSV or Excel dataset.")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No dataset uploaded. Please upload a CSV or Excel file.")

file_name = next(iter(uploaded))

if file_name.lower().endswith(".csv"):
    try:
        df = pd.read_csv(file_name, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(file_name, encoding="latin1")

elif file_name.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(file_name)

else:
    raise ValueError("Unsupported file. Please upload CSV or Excel.")

df.columns = df.columns.astype(str).str.strip()

# Remove completely empty rows
df.dropna(how="all", inplace=True)
df.reset_index(drop=True, inplace=True)

# Preserve original dataset
raw_df = df.copy(deep=True)

original_rows = len(raw_df)
original_columns = len(raw_df.columns)

# Change log storage
change_log = []

def log_change(description, impact, status="Completed"):
    change_log.append({
        "Change ID": f"CR-{len(change_log) + 1:03d}",
        "Description": str(description),
        "Impact": str(impact),
        "Status": status
    })

print("\nDataset uploaded successfully!")
print("Dataset:", file_name)
print("Original rows:", original_rows)
print("Original columns:", original_columns)

# =====================================================================
# 5. INITIAL DATASET INSPECTION
# =====================================================================

print("\n" + "=" * 70)
print("INITIAL DATASET INSPECTION")
print("=" * 70)

print("\nFirst five records:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

# =====================================================================
# 6. IDENTIFY ID AND DATE COLUMNS
# =====================================================================

# Add your custom ID column names here if needed.
# Example: ID_COLUMNS = ["order_id", "customer_id"]

ID_COLUMNS = [
    col for col in df.columns
    if col.lower().strip() in [
        "id", "order_id", "customer_id",
        "transaction_id", "record_id",
        "employee_id", "product_id",
        "user_id", "student_id"
    ]
]

# Add custom date column names here if needed.
DATE_COLUMNS = [
    col for col in df.columns
    if any(word in col.lower() for word in [
        "date", "timestamp", "datetime"
    ])
]

print("\nDetected ID columns:", ID_COLUMNS)
print("Detected date columns:", DATE_COLUMNS)

# Convert ID columns to consistent text identifiers.
for col in ID_COLUMNS:

    def normalize_id(value):
        if pd.isna(value):
            return pd.NA

        if isinstance(value, (int, np.integer)):
            return str(value)

        if isinstance(value, (float, np.floating)):
            if value.is_integer():
                return str(int(value))

        return str(value).strip()

    df[col] = df[col].apply(normalize_id).astype("string")

# =====================================================================
# 7. IDENTIFY MISSING VALUES
# =====================================================================

print("\n" + "=" * 70)
print("MISSING VALUE ANALYSIS")
print("=" * 70)

missing_before = df.isnull().sum()

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": missing_before.values,
    "Missing Percentage": (
        missing_before.values / max(len(df), 1) * 100
    ).round(2)
})

display(missing_report)

print("Total missing values:", int(missing_before.sum()))

# =====================================================================
# 8. REMOVE COMPLETELY EMPTY COLUMNS
# =====================================================================

empty_columns = df.columns[df.isna().all()].tolist()

if empty_columns:

    df.drop(columns=empty_columns, inplace=True)

    log_change(
        "Removed completely empty columns",
        f"{len(empty_columns)} empty columns removed."
    )

    ID_COLUMNS = [c for c in ID_COLUMNS if c in df.columns]
    DATE_COLUMNS = [c for c in DATE_COLUMNS if c in df.columns]

# =====================================================================
# 9. REMOVE DUPLICATE RECORDS
# =====================================================================

print("\n" + "=" * 70)
print("DUPLICATE RECORD IDENTIFICATION")
print("=" * 70)

duplicate_count = int(df.duplicated().sum())

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print("Duplicate records removed:", duplicate_count)

if duplicate_count > 0:
    log_change(
        "Removed completely duplicated records",
        f"{duplicate_count} duplicate records removed."
    )

# =====================================================================
# 10. VALIDATE UNIQUE IDENTIFIERS
# =====================================================================

print("\n" + "=" * 70)
print("UNIQUE ID VALIDATION")
print("=" * 70)

for col in ID_COLUMNS:

    missing_ids = int(df[col].isna().sum())

    if missing_ids > 0:

        df.dropna(subset=[col], inplace=True)

        log_change(
            f"Removed records with missing {col}",
            f"{missing_ids} records removed because IDs were missing."
        )

    duplicate_ids = int(df[col].duplicated().sum())

    if duplicate_ids > 0:

        df.drop_duplicates(
            subset=[col], keep="first", inplace=True
        )

        log_change(
            f"Removed duplicate identifiers in {col}",
            f"{duplicate_ids} duplicate IDs removed; first record retained."
        )

    print(
        f"{col}: {df[col].nunique()} unique IDs"
    )

df.reset_index(drop=True, inplace=True)

# =====================================================================
# 11. DATE VALIDATION AND MISSING DATE HANDLING
# =====================================================================

print("\n" + "=" * 70)
print("DATE VALIDATION")
print("=" * 70)

for col in DATE_COLUMNS:

    original_present = df[col].notna()

    try:
        parsed_dates = pd.to_datetime(
            df[col],
            format="mixed",
            dayfirst=True,
            errors="coerce"
        )
    except (TypeError, ValueError):
        parsed_dates = pd.to_datetime(
            df[col],
            dayfirst=True,
            errors="coerce"
        )

    invalid_count = int(
        (original_present & parsed_dates.isna()).sum()
    )

    missing_count = int(parsed_dates.isna().sum())

    if missing_count > 0:

        df = df.loc[parsed_dates.notna()].copy()
        parsed_dates = parsed_dates.loc[df.index]

        log_change(
            f"Removed missing or invalid dates from {col}",
            f"{missing_count} unreliable date records removed."
        )

    df[col] = parsed_dates.dt.strftime("%Y-%m-%d")

    log_change(
        f"Standardized {col} to ISO 8601",
        f"Dates converted to YYYY-MM-DD; {invalid_count} invalid values detected."
    )

    print(f"{col}: Date format standardized.")

# =====================================================================
# 12. STRATEGIC IMPUTATION
# MEAN + MEDIAN + MODE
# =====================================================================

print("\n" + "=" * 70)
print("STRATEGIC MISSING VALUE IMPUTATION")
print("=" * 70)

# Explicit strategy configuration.
# Mean: suitable numerical columns such as quantity.
# Median: numerical columns such as sales_amount.
# Mode: categorical columns.

IMPUTATION_METHODS = {
    "quantity": "mean",
    "qty": "mean",
    "units": "mean",
    "sales_amount": "median",
    "price": "median",
    "revenue": "median"
}

for col in df.columns:

    if col in ID_COLUMNS or col in DATE_COLUMNS:
        continue

    missing_count = int(df[col].isna().sum())

    if missing_count == 0:
        continue

    if pd.api.types.is_numeric_dtype(df[col]):

        method = IMPUTATION_METHODS.get(
            col.lower(), "median"
        )

        if method == "mean":
            fill_value = df[col].mean()
        else:
            fill_value = df[col].median()

        if pd.notna(fill_value):

            df[col] = df[col].fillna(fill_value)

            log_change(
                f"{method.capitalize()} imputation in {col}",
                f"{missing_count} missing values filled using {method}."
            )

            print(
                f"{col}: {missing_count} missing values filled using {method}."
            )

    else:

        mode_values = df[col].mode(dropna=True)

        if not mode_values.empty:
            fill_value = mode_values.iloc[0]
        else:
            fill_value = "Unknown"

        df[col] = df[col].fillna(fill_value)

        log_change(
            f"Mode imputation in {col}",
            f"{missing_count} missing values filled using mode."
        )

        print(
            f"{col}: {missing_count} missing values filled using mode."
        )

# =====================================================================
# 13. TEXT STANDARDIZATION
# TRIM WHITESPACE + PROPER CASE + CITY NORMALIZATION
# =====================================================================

print("\n" + "=" * 70)
print("TEXT STANDARDIZATION")
print("=" * 70)

for col in df.select_dtypes(
    include=["object", "string"]
).columns:

    before = df[col].copy()

    # Remove leading/trailing and repeated spaces.
    df[col] = (
        df[col].astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Standardize city names.
    if col.lower() in [
        "city", "location", "customer_city"
    ]:

        city_mapping = {
            "bangalore": "Bengaluru",
            "bengaluru": "Bengaluru",
            "blore": "Bengaluru",
            "bombay": "Mumbai",
            "mumbai": "Mumbai",
            "pondicherry": "Puducherry",
            "puducherry": "Puducherry"
        }

        normalized = df[col].str.lower().map(city_mapping)

        df[col] = normalized.fillna(df[col])

    # Proper case for human-readable text columns.
    elif any(word in col.lower() for word in [
        "name", "category", "type", "department",
        "product", "status"
    ]):

        df[col] = df[col].str.title()

    changed = int(
        (before.fillna("") != df[col].fillna("")).sum()
    )

    if changed > 0:

        log_change(
            f"Text standardization in {col}",
            f"{changed} text values trimmed or standardized."
        )

    print(f"{col}: Text standardization checked.")

print("Text cleaning completed.")

# =====================================================================
# 14. NUMERIC PRECISION STANDARDIZATION
# =====================================================================

print("\n" + "=" * 70)
print("NUMERIC PRECISION STANDARDIZATION")
print("=" * 70)

numeric_columns = df.select_dtypes(
    include=["number"]
).columns.tolist()

for col in numeric_columns:

    if col in ID_COLUMNS:
        continue

    df[col] = df[col].round(2)

    log_change(
        f"Numeric precision correction in {col}",
        "Values rounded to 2 decimal places."
    )

    print(f"{col}: Rounded to 2 decimal places.")

# =====================================================================
# 15. FINAL MISSING VALUE TREATMENT
# =====================================================================

remaining_missing = int(df.isnull().sum().sum())

print("\nRemaining missing values:", remaining_missing)

# If an entirely empty column remains, it cannot be imputed.
# The audit will report the issue rather than inventing data.

# =====================================================================
# 16. BEFORE AND AFTER COMPARISON
# =====================================================================

print("\n" + "=" * 70)
print("BEFORE AND AFTER COMPARISON")
print("=" * 70)

comparison = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Missing Values",
        "Duplicate Rows"
    ],

    "Before Cleaning": [
        len(raw_df),
        len(raw_df.columns),
        int(raw_df.isnull().sum().sum()),
        int(raw_df.duplicated().sum())
    ],

    "After Cleaning": [
        len(df),
        len(df.columns),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum())
    ]
})

display(comparison)

# =====================================================================
# 17. FINAL INTEGRITY AUDIT
# =====================================================================

print("\n" + "=" * 70)
print("FINAL DATA INTEGRITY AUDIT")
print("=" * 70)

# Duplicate rows
duplicate_rows_remaining = int(df.duplicated().sum())

# Missing values
missing_values_remaining = int(df.isnull().sum().sum())

# Missing and duplicate IDs
duplicate_ids_remaining = 0
missing_ids_remaining = 0

for col in ID_COLUMNS:

    duplicate_ids_remaining += int(df[col].duplicated().sum())
    missing_ids_remaining += int(df[col].isna().sum())

# Date validation
invalid_dates_remaining = 0

for col in DATE_COLUMNS:

    values = df[col].astype("string")

    correct_format = values.str.fullmatch(
        r"\d{4}-\d{2}-\d{2}",
        na=False
    )

    actual_dates = pd.to_datetime(
        values,
        format="%Y-%m-%d",
        errors="coerce"
    )

    valid_dates = correct_format & actual_dates.notna()

    invalid_dates_remaining += int((~valid_dates).sum())

# Audit results
audit_results = {
    "Duplicate Rows": duplicate_rows_remaining,
    "Duplicate IDs": duplicate_ids_remaining,
    "Missing IDs": missing_ids_remaining,
    "Remaining Missing Values": missing_values_remaining,
    "Incorrect Date Formats": invalid_dates_remaining
}

audit_rows = []

for check, result in audit_results.items():

    status = "PASS" if result == 0 else "FAIL"

    audit_rows.append({
        "Audit Check": check,
        "Remaining Issues": result,
        "Result": status
    })

    print(f"{check}: {result} | {status}")

if not ID_COLUMNS:
    print("\nWARNING: No ID column detected. Add its exact name to ID_COLUMNS.")

if not DATE_COLUMNS:
    print("\nNOTE: No date columns detected. Date audit was not applicable.")

# =====================================================================
# 18. SAVE CLEANED DATASET
# =====================================================================

cleaned_file = "DecodeLabs_Cleaned_Dataset.csv"

df.to_csv(cleaned_file, index=False)

print("\nCleaned dataset saved successfully!")

# =====================================================================
# 19. GENERATE PROFESSIONAL PDF CHANGE LOG
# REQUIRED FIELDS:
# CHANGE ID | DESCRIPTION | IMPACT | STATUS
# =====================================================================

pdf_file = "DecodeLabs_Project1_Change_Log.pdf"

styles = getSampleStyleSheet()

styles["Title"].alignment = TA_CENTER

story = []

story.append(Paragraph(
    "DECODELABS",
    styles["Title"]
))

story.append(Paragraph(
    "DATA ANALYTICS - PROJECT 1",
    styles["Heading1"]
))

story.append(Paragraph(
    "DATA CLEANING & PREPARATION REPORT",
    styles["Heading2"]
))

story.append(Spacer(1, 15))

summary = f"""
<b>Dataset Name:</b> {file_name}<br/>
<b>Execution Date:</b> {datetime.now().strftime('%Y-%m-%d %H:%M')}<br/>
<b>Original Rows:</b> {original_rows}<br/>
<b>Final Rows:</b> {len(df)}<br/>
<b>Original Columns:</b> {original_columns}<br/>
<b>Final Columns:</b> {len(df.columns)}<br/>
<b>ID Columns:</b> {', '.join(ID_COLUMNS) if ID_COLUMNS else 'Not detected'}<br/>
<b>Date Columns:</b> {', '.join(DATE_COLUMNS) if DATE_COLUMNS else 'Not detected'}
"""

story.append(Paragraph(summary, styles["BodyText"]))

story.append(Spacer(1, 20))

story.append(Paragraph(
    "CHANGE LOG",
    styles["Heading2"]
))

change_data = [[
    "Change ID",
    "Description",
    "Impact",
    "Status"
]]

for item in change_log:

    change_data.append([
        item["Change ID"],
        Paragraph(item["Description"], styles["BodyText"]),
        Paragraph(item["Impact"], styles["BodyText"]),
        item["Status"]
    ])

if not change_log:

    change_data.append([
        "CR-000",
        "No changes required",
        "Dataset already met cleaning conditions.",
        "Verified"
    ])

change_table = Table(
    change_data,
    colWidths=[60, 185, 260, 65],
    repeatRows=1
)

change_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0),
     colors.HexColor("#17324D")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("LEFTPADDING", (0, 0), (-1, -1), 7),
    ("RIGHTPADDING", (0, 0), (-1, -1), 7),
    ("TOPPADDING", (0, 0), (-1, -1), 7),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1),
     [colors.white, colors.HexColor("#EAF0F5")])
]))

story.append(change_table)

story.append(Spacer(1, 20))

# Final audit table
story.append(Paragraph(
    "FINAL INTEGRITY AUDIT",
    styles["Heading2"]
))

audit_data = [[
    "Audit Check",
    "Remaining Issues",
    "Result"
]]

for item in audit_rows:

    audit_data.append([
        item["Audit Check"],
        str(item["Remaining Issues"]),
        item["Result"]
    ])

audit_table = Table(
    audit_data,
    colWidths=[250, 150, 100]
)

audit_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0),
     colors.HexColor("#17324D")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTSIZE", (0, 0), (-1, -1), 9),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("PADDING", (0, 0), (-1, -1), 8)
]))

story.append(audit_table)

story.append(Spacer(1, 20))

story.append(Paragraph(
    "PROJECT CONCLUSION",
    styles["Heading2"]
))

story.append(Paragraph(
    "The dataset was processed using Python and Pandas. "
    "Data cleaning operations included missing value treatment, "
    "duplicate removal, identifier validation, date standardization, "
    "text standardization and numeric precision correction. "
    "All recorded changes are documented in this report. "
    "The final integrity audit provides the verification results "
    "for the cleaned dataset.",
    styles["BodyText"]
))

pdf = SimpleDocTemplate(
    pdf_file,
    pagesize=landscape(A4),
    rightMargin=30,
    leftMargin=30,
    topMargin=30,
    bottomMargin=30
)

pdf.build(story)

print("\nPDF Change Log generated successfully!")

# =====================================================================
# 20. FINAL PROJECT SUMMARY
# =====================================================================

print("\n" + "=" * 70)
print("DECODELABS PROJECT 1 - EXECUTION SUMMARY")
print("=" * 70)

print("Original dataset rows:", original_rows)
print("Cleaned dataset rows:", len(df))
print("Total changes documented:", len(change_log))
print("Final dataset shape:", df.shape)

display(pd.DataFrame(audit_rows))

# Check if all applicable audit conditions passed
all_passed = all(
    item["Result"] == "PASS"
    for item in audit_rows
)

print("\nFINAL AUDIT STATUS:",
      "ALL CHECKS PASSED" if all_passed else "REVIEW REQUIRED")

# Save audit results separately as CSV too
audit_file = "DecodeLabs_Final_Audit.csv"

pd.DataFrame(audit_rows).to_csv(
    audit_file, index=False
)

# =====================================================================
# 21. DOWNLOAD ALL PROJECT OUTPUTS
# =====================================================================

print("\nPreparing project files for download...")

for output_file in [
    cleaned_file,
    pdf_file,
    audit_file
]:

    print("Downloading:", output_file)
    files.download(output_file)

print("\n" + "=" * 70)
print("PROJECT 1 PROCESSING FINISHED")
print("Thank you - DecodeLabs Data Analytics")
print("=" * 70)


Installing required libraries...
All libraries installed and imported successfully!

DECODELABS - DATA ANALYTICS PROJECT 1
DATA CLEANING & PREPARATION

Upload your raw CSV or Excel dataset.


Saving Dataset for Data Analytics.xlsx to Dataset for Data Analytics (1).xlsx

Dataset uploaded successfully!
Dataset: Dataset for Data Analytics (1).xlsx
Original rows: 1200
Original columns: 14

INITIAL DATASET INSPECTION

First five records:


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
OrderID,1200,1200,ORD201183,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,1200,NaN,NaN,NaN,2024-03-22 16:58:48,2023-01-01 00:00:00,2023-08-03 18:00:00,2024-03-23 00:00:00,2024-11-08 12:00:00,2025-06-30 00:00:00,NaN
CustomerID,1200,1189,C46651,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product,1200,7,Printer,181,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,1200.0,NaN,NaN,NaN,2.945833,1.0,2.0,3.0,4.0,5.0,1.407557
UnitPrice,1200.0,NaN,NaN,NaN,356.41275,11.39,186.0625,364.21,521.57,699.93,197.177146
ShippingAddress,1200,655,533 Main St,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PaymentMethod,1200,5,Online,258,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OrderStatus,1200,5,Cancelled,250,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TrackingNumber,1200,1200,TRK83653396,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Detected ID columns: []
Detected date columns: ['Date']

MISSING VALUE ANALYSIS


... (11 outputs left out)

Downloading: DecodeLabs_Final_Audit.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PROJECT 1 PROCESSING FINISHED
Thank you - DecodeLabs Data Analytics
